# 예제 02. `DataProfiler` — 규칙 기반 진단기

전산실험 2 · AI Agent 만들기 · **9주차** · 예상 소요 35분

## 목표
- **LLM 없이** 규칙만으로 데이터를 진단한다
- 사람용 보고서와 LLM용 요약을 나눈다
- 무엇을 고쳤는지 **전부 출력**하는 `clean()` 을 쓴다
- 규칙과 LLM의 **경계**를 판단한다

### 준비 — 공용 모듈과 데이터 만들기 / Setup — write the shared modules and data

이 수업 저장소에는 노트북(`.ipynb`)만 둡니다. 그래서 실습에 필요한 공용 모듈과
데이터를 **아래 셀들이 직접 파일로 만들어 냅니다.** 내려받지 않으므로 네트워크가
느려도, 저장소가 비공개로 바뀌어도 그대로 동작합니다.

아래 셀을 **위에서부터 차례로 한 번씩** 실행한 뒤 다음으로 넘어가세요.
내용을 지금 읽을 필요는 없습니다. 만들어진 `.py` 파일은 왼쪽 파일 탐색기에서
언제든 열어 볼 수 있습니다.

The course repository contains notebooks only. The cells below **write the shared
modules and data files themselves** instead of downloading them. Run them once,
top to bottom, then continue. You do not need to read them now — the generated
`.py` files can be opened from the file browser on the left.

In [ ]:
%%writefile kmu_llm.py
"""전산실험2 - AI Agent 만들기 : 공용 LLM 래퍼.

계명대 통계학과 서버와 Google Gemini를 같은 방식으로 호출한다.
둘 다 OpenAI 호환 API를 제공하므로 openai SDK 하나로 통일하고
base_url 과 model 만 바꾼다.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 이 저장소는 공개(public)이므로 **학과 서버 주소와 API 키를 코드에 적지
 않는다.** 둘 다 수업시간에 공개하고, 각자 Colab Secrets 에 저장해서 쓴다.

     KMU_BASE_URL  = 수업시간에 공개  (예: https://.../v1)
     KMU_API_KEY   = 수업시간에 공개

 왼쪽 열쇠 아이콘(🔑) → 새 secret → 노트북 액세스 토글 켜기
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

사용 예:
    from kmu_llm import LLM, ask, check_server

    check_server()                       # 서버가 살아 있는지 먼저 확인
    print(ask("표준편차를 한 문장으로 설명해줘"))

    llm = LLM(provider="gemini")         # 백업 경로
    print(llm.chat([{"role": "user", "content": "안녕"}]))
"""

from __future__ import annotations

import os
import time
from typing import Any, Iterator

__all__ = [
    "KMU_MODEL",
    "GEMINI_BASE_URL",
    "GEMINI_MODEL",
    "PROVIDERS",
    "LLM",
    "ask",
    "get_secret",
    "get_key",
    "get_base_url",
    "check_server",
]

# --------------------------------------------------------------------------
# 서버 설정
# --------------------------------------------------------------------------
# 학과 서버 주소는 여기에 적지 않는다. Colab Secrets 의 KMU_BASE_URL 에서 읽는다.
# 모델 태그는 공개해도 무방하므로 상수로 둔다.
KMU_MODEL = os.environ.get("KMU_MODEL", "gemma4")

# 백업 경로. Google AI Studio 에서 무료 키를 발급받아 쓴다. (공개 주소)
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GEMINI_MODEL = "gemini-2.0-flash"

PROVIDERS: dict[str, dict[str, Any]] = {
    "kmu": {
        "base_url": None,  # 실행 시점에 Secrets 에서 읽는다
        "base_url_key": "KMU_BASE_URL",
        "model": KMU_MODEL,
        "key_name": "KMU_API_KEY",
        "label": "계명대 학과 서버",
    },
    "gemini": {
        "base_url": GEMINI_BASE_URL,
        "base_url_key": None,
        "model": GEMINI_MODEL,
        "key_name": "GEMINI_API_KEY",
        "label": "Google Gemini (백업)",
    },
}


# --------------------------------------------------------------------------
# 비밀정보 가져오기
# --------------------------------------------------------------------------
def get_secret(name: str, *, prompt: bool = True, secret: bool = True) -> str:
    """설정값을 안전한 순서로 찾는다.

    1) Colab Secrets (왼쪽 열쇠 아이콘)  2) 환경변수  3) 직접 입력

    값을 코드에 적지 않는 것이 핵심이다. 노트북이나 모듈을 GitHub 에 올리는
    순간 코드에 적힌 값은 전 세계에 공개된다. API 키뿐 아니라 **내부 서버
    주소도** 마찬가지다.

    Parameters
    ----------
    secret : True 면 입력을 화면에 표시하지 않는다(getpass).
             주소처럼 오타 확인이 중요한 값은 False 로 준다.
    """
    # 1) Colab Secrets
    try:
        from google.colab import userdata  # type: ignore

        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass

    # 2) 환경변수
    value = os.environ.get(name)
    if value:
        return value.strip()

    # 3) 직접 입력
    if not prompt:
        raise RuntimeError(
            f"'{name}' 를 찾을 수 없습니다.\n"
            f"  Colab 왼쪽 열쇠 아이콘(🔑) → 새 secret → 이름 '{name}'\n"
            f"  값은 수업시간에 공개합니다. 노트북 액세스 토글도 켜세요."
        )

    print(f"[{name}] 를 찾지 못했습니다. 직접 입력해 주세요.")
    print(f"   (Colab Secrets 에 '{name}' 로 저장해 두면 다시 묻지 않습니다)")

    if secret:
        import getpass

        value = getpass.getpass(f"{name}: ").strip()
    else:
        value = input(f"{name}: ").strip()

    if not value:
        raise RuntimeError(f"'{name}' 에 빈 값이 입력되었습니다.")
    os.environ[name] = value  # 같은 런타임 안에서는 다시 묻지 않는다
    return value


def get_key(key_name: str = "KMU_API_KEY", *, prompt: bool = True) -> str:
    """API 키를 가져온다. (get_secret 의 별칭 — 1주차 이름 유지)"""
    return get_secret(key_name, prompt=prompt, secret=True)


def get_base_url(provider: str = "kmu", *, prompt: bool = True) -> str:
    """서버 주소를 가져온다. 끝의 '/' 는 떼고 돌려준다.

    학과 서버 주소는 저장소에 적지 않으므로 Secrets 에서 읽는다.
    '/chat/completions' 까지 붙여서 저장한 경우도 알아서 잘라낸다.
    """
    conf = PROVIDERS[provider]

    if conf["base_url"]:
        return str(conf["base_url"]).rstrip("/")

    url = get_secret(conf["base_url_key"], prompt=prompt, secret=False).rstrip("/")

    # openai SDK 는 base_url 뒤에 /chat/completions 를 스스로 붙인다.
    # 주소를 통째로 저장한 경우를 대비해 정리해 준다.
    for suffix in ("/chat/completions", "/completions"):
        if url.endswith(suffix):
            url = url[: -len(suffix)]

    conf["base_url"] = url  # 같은 세션에서는 재사용
    return url


# --------------------------------------------------------------------------
# LLM 클라이언트
# --------------------------------------------------------------------------
class LLM:
    """OpenAI 호환 서버를 감싼 얇은 클라이언트.

    Parameters
    ----------
    provider : "kmu" | "gemini"
        어느 서버를 쓸지. 기본은 학과 서버.
    model, base_url, api_key :
        직접 지정하면 기본값을 덮어쓴다. (보통 지정하지 않는다)
    """

    def __init__(
        self,
        provider: str = "kmu",
        model: str | None = None,
        base_url: str | None = None,
        api_key: str | None = None,
        timeout: float = 120.0,
    ) -> None:
        if provider not in PROVIDERS:
            raise ValueError(
                f"알 수 없는 provider: {provider!r}. 가능한 값: {list(PROVIDERS)}"
            )
        conf = PROVIDERS[provider]

        self.provider = provider
        self.label = conf["label"]
        self.model = model or conf["model"]
        self.base_url = base_url or get_base_url(provider)
        self.api_key = api_key or get_key(conf["key_name"])

        try:
            from openai import OpenAI
        except ImportError as exc:  # pragma: no cover
            raise ImportError(
                "openai 패키지가 필요합니다.  !pip install -q openai"
            ) from exc

        self.client = OpenAI(
            api_key=self.api_key, base_url=self.base_url, timeout=timeout
        )

    def __repr__(self) -> str:
        # 주소는 출력하지 않는다. 화면 공유·스크린샷으로 새어나가기 쉽다.
        return f"LLM(provider={self.provider!r}, model={self.model!r})"

    # -- 기본 호출 --------------------------------------------------------
    def raw(self, messages: list[dict[str, str]], **kw: Any):
        """응답 객체 전체를 그대로 돌려준다. usage(토큰 수) 확인용."""
        params: dict[str, Any] = {
            "model": self.model,
            "messages": messages,
            "temperature": 0.7,
            "max_tokens": 1024,
        }
        params.update(kw)
        return self.client.chat.completions.create(**params)

    def chat(self, messages: list[dict[str, str]], **kw: Any) -> str:
        """대화 메시지를 보내고 응답 '텍스트'만 돌려준다."""
        response = self.raw(messages, **kw)
        content = response.choices[0].message.content
        return (content or "").strip()

    def stream(self, messages: list[dict[str, str]], **kw: Any) -> Iterator[str]:
        """응답을 조각(chunk) 단위로 흘려보낸다. 타이핑되는 효과."""
        kw["stream"] = True
        for chunk in self.raw(messages, **kw):
            if not chunk.choices:
                continue
            piece = chunk.choices[0].delta.content
            if piece:
                yield piece

    # -- 편의 메서드 ------------------------------------------------------
    def ask(self, prompt: str, system: str | None = None, **kw: Any) -> str:
        """한 번짜리 질문. messages 를 매번 만들기 번거로울 때 쓴다."""
        messages: list[dict[str, str]] = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        return self.chat(messages, **kw)

    def ask_retry(
        self,
        prompt: str,
        system: str | None = None,
        *,
        n_retry: int = 3,
        wait: float = 2.0,
        **kw: Any,
    ) -> str:
        """일시적인 네트워크·서버 오류를 재시도한다.

        30회 반복 실험처럼 호출이 많은 실습에서 한 번의 실패로 전체가
        멈추지 않도록 하기 위한 것이다.
        """
        last_error: Exception | None = None
        for attempt in range(1, n_retry + 1):
            try:
                return self.ask(prompt, system, **kw)
            except Exception as exc:  # 네트워크·서버 오류 전반
                last_error = exc
                if attempt < n_retry:
                    print(f"   [재시도 {attempt}/{n_retry - 1}] {type(exc).__name__}")
                    time.sleep(wait * attempt)  # 점점 더 오래 기다린다
        raise RuntimeError(f"{n_retry}회 시도 모두 실패했습니다.") from last_error

    def count_tokens(self, messages: list[dict[str, str]]) -> int:
        """입력 토큰 수를 서버에 물어본다. 출력은 1토큰으로 잘라 낭비를 줄인다."""
        return self.raw(messages, max_tokens=1).usage.prompt_tokens


# --------------------------------------------------------------------------
# 모듈 수준 편의 함수
# --------------------------------------------------------------------------
_default_llm: LLM | None = None


def ask(
    prompt: str, system: str | None = None, *, provider: str = "kmu", **kw: Any
) -> str:
    """가장 간단한 한 줄 호출.  ask("안녕") -> "안녕하세요..."

    내부적으로 LLM 객체를 한 번만 만들어 재사용한다.
    """
    global _default_llm
    if _default_llm is None or _default_llm.provider != provider:
        _default_llm = LLM(provider=provider)
    return _default_llm.ask(prompt, system, **kw)


# --------------------------------------------------------------------------
# 서버 진단
# --------------------------------------------------------------------------
def check_server(provider: str = "kmu", base_url: str | None = None) -> dict[str, Any]:
    """서버가 살아 있는지, 어떤 모델이 올라가 있는지 확인한다.

    수업 첫 실습. 실패하면 무엇을 확인해야 하는지 함께 출력한다.

    Returns
    -------
    dict : {"ok": bool, "models": list[str], "error": str | None}
    """
    import requests

    conf = PROVIDERS[provider]
    base_url = (base_url or get_base_url(provider)).rstrip("/")
    api_key = get_key(conf["key_name"])

    # 주소 전체를 출력하지 않는다. 호스트만 가려서 보여준다.
    print(f"조회 중: GET {_mask_url(base_url)}/models")

    try:
        response = requests.get(
            f"{base_url}/models",
            headers={"Authorization": f"Bearer {api_key}"},
            timeout=15,
        )
    except Exception as exc:
        print(f"\n[실패] 서버에 연결하지 못했습니다: {type(exc).__name__}")
        _print_troubleshooting()
        return {"ok": False, "models": [], "error": str(exc)}

    if response.status_code != 200:
        print(f"\n[실패] HTTP {response.status_code}")
        print(f"   응답: {response.text[:300]}")
        if response.status_code in (401, 403):
            print("   -> API 키가 잘못되었거나 만료되었을 가능성이 높습니다.")
        elif response.status_code == 404:
            print("   -> KMU_BASE_URL 이 '/v1' 로 끝나는지 확인하세요.")
        return {"ok": False, "models": [], "error": f"HTTP {response.status_code}"}

    models = [m.get("id", "?") for m in response.json().get("data", [])]

    print(f"\n[성공] 서버가 응답했습니다. 사용 가능한 모델 {len(models)}개:")
    for name in models:
        mark = "  <- 기본 모델" if name == conf["model"] else ""
        print(f"   - {name}{mark}")

    if conf["model"] not in models and models:
        print(
            f"\n[주의] 설정된 모델 '{conf['model']}' 이 목록에 없습니다.\n"
            f"       위 목록에서 골라 LLM(model='...') 로 지정하거나\n"
            f"       환경변수 KMU_MODEL 을 설정하세요."
        )

    return {"ok": True, "models": models, "error": None}


def _mask_url(url: str) -> str:
    """화면 공유·스크린샷 유출을 막기 위해 호스트를 가린다."""
    import re

    return re.sub(r"//([^/]+)", lambda m: "//" + m.group(1)[:4] + "***", url, count=1)


def _print_troubleshooting() -> None:
    """연결 실패 시 확인할 것들."""
    print("\n확인해 볼 것:")
    print("   1. Colab Secrets 의 KMU_BASE_URL 값이 정확합니까?")
    print("      - http / https 를 맞게 적었습니까?")
    print("      - 포트가 필요할 수 있습니다.")
    print("      - 끝이 '/v1' 이어야 합니다. ('/chat/completions' 는 빼도 됩니다)")
    print("   2. 학과 서버가 켜져 있습니까? 교내망에서만 열려 있지 않습니까?")
    print("   3. 다른 주소를 시험해 보려면:")
    print("      check_server(base_url='...')")
    print("   4. 계속 안 되면 백업 경로를 쓰세요:  LLM(provider='gemini')")


In [ ]:
%%writefile profiler.py
"""전산실험2 9주차 : 데이터 진단과 분석계획 수립.

7주차까지 우리는 **깨끗한 데이터를 전제**했다. 실제 데이터는 그렇지 않다.
`-999` 가 결측이고, `'12.5kg'` 이 수치이고, `Yes/yes/Y` 가 같은 값이다.

이 모듈은 두 가지를 한다.

    DataProfiler   데이터를 훑어 타입을 추론하고 문제를 찾아낸다 (규칙 기반)
    AnalysisPlanner LLM이 분석 계획을 세우되, **규칙이 검증한다**

핵심 설계 원칙:
    **무엇을 LLM에게 맡기고 무엇을 규칙으로 고정할 것인가.**

    규칙으로 고정  : 검정의 전제 조건 (집단 수, 대응 여부, 척도)
                    -> 틀리면 결과 자체가 무의미하므로 코드가 강제한다
    LLM에게 위임   : 변수의 의미 파악, 사용자 질문 해석, 결과 서술
                    -> 규칙으로 적을 수 없는 판단

사용 예:
    import pandas as pd
    from profiler import DataProfiler

    prof = DataProfiler(pd.read_csv("exam_greenhouse.csv"))
    print(prof.report())
    clean = prof.clean()
"""

from __future__ import annotations

import re
from dataclasses import dataclass, field
from typing import Any

__all__ = [
    "SENTINELS",
    "ColumnProfile",
    "DataProfiler",
    "TEST_RULES",
    "validate_plan",
    "AnalysisPlanner",
]

# 결측을 뜻하는 관례적 값들. 센서·설문 데이터에서 흔하다.
SENTINELS = {-999, -9999, -99, 999, 9999, -1e30}
SENTINEL_STRINGS = {"", "na", "n/a", "null", "none", "nan", "-", "?", "결측", "무응답"}

_UNIT_SUFFIX = re.compile(r"^\s*([-+]?[\d.,]+)\s*[a-zA-Z가-힣%°]+\s*$")
_DECIMAL_COMMA = re.compile(r"^\s*[-+]?\d{1,3}(,\d{1,2})\s*$")

# 같은 뜻인데 표기가 다른 값들. 설문·수기 입력에서 늘 나온다.
# 축약형(y/n)과 완전형(yes/no)이 한 열에 섞이면 대소문자 정규화만으로는
# 합쳐지지 않는다.
SYNONYM_FAMILIES: list[dict[str, str]] = [
    {"y": "yes", "yes": "yes", "true": "yes", "t": "yes", "1": "yes",
     "n": "no", "no": "no", "false": "no", "f": "no", "0": "no",
     "예": "yes", "아니오": "no", "아니요": "no"},
    {"m": "male", "male": "male", "남": "male", "남자": "male", "남성": "male",
     "f": "female", "female": "female", "여": "female", "여자": "female",
     "여성": "female"},
]


# --------------------------------------------------------------------------
# 열 하나의 진단 결과
# --------------------------------------------------------------------------
@dataclass
class ColumnProfile:
    name: str
    dtype: str
    n: int
    n_missing: int = 0
    n_unique: int = 0
    kind: str = "unknown"          # continuous / ordinal / nominal / id / binary
    issues: list[str] = field(default_factory=list)
    fixes: list[str] = field(default_factory=list)   # 적용 가능한 정리 방법
    stats: dict[str, Any] = field(default_factory=dict)
    levels: list[str] = field(default_factory=list)

    def line(self) -> str:
        문제 = f"  ⚠ {'; '.join(self.issues)}" if self.issues else ""
        return (f"- {self.name} [{self.kind}] "
                f"n={self.n - self.n_missing}/{self.n}, 고유 {self.n_unique}{문제}")


# --------------------------------------------------------------------------
# 데이터 진단
# --------------------------------------------------------------------------
class DataProfiler:
    """DataFrame 을 훑어 타입을 추론하고 문제를 찾아낸다.

    **전부 규칙 기반이다. LLM을 쓰지 않는다.**
    타입 추론과 결측 탐지는 규칙으로 충분히 정확하고,
    LLM에게 맡기면 느리고 비싸고 틀린다.
    """

    def __init__(self, df: Any, *, ordinal_max_levels: int = 10) -> None:
        self.df = df
        self.ordinal_max_levels = ordinal_max_levels
        self.profiles: dict[str, ColumnProfile] = {}
        self._scan()

    # ----------------------------------------------------------------
    def _scan(self) -> None:
        import pandas as pd

        n = len(self.df)
        for col in self.df.columns:
            s = self.df[col]
            p = ColumnProfile(name=col, dtype=str(s.dtype), n=n,
                              n_missing=int(s.isna().sum()),
                              n_unique=int(s.nunique(dropna=True)))

            문자 = s.dropna().astype(str)

            # --- 수치로 볼 수 있는가 -------------------------------
            숫자 = pd.to_numeric(s, errors="coerce")
            변환실패 = int(숫자.isna().sum() - s.isna().sum())
            수치비율 = 숫자.notna().sum() / max(n - p.n_missing, 1)

            # --- 문자열에 숨은 수치 패턴 ---------------------------
            단위붙음 = int(문자.apply(lambda v: bool(_UNIT_SUFFIX.match(v))).sum())
            쉼표소수 = int(문자.apply(lambda v: bool(_DECIMAL_COMMA.match(v))).sum())

            if 단위붙음:
                p.issues.append(f"단위가 붙은 값 {단위붙음}건 (예: '12.5kg')")
                p.fixes.append("strip_units")
            if 쉼표소수:
                p.issues.append(f"소수점에 쉼표를 쓴 값 {쉼표소수}건 (예: '3,5')")
                p.fixes.append("comma_decimal")

            수치가능 = 수치비율 > 0.7 or (단위붙음 + 쉼표소수) > 0

            # --- 결측 위장값 ---------------------------------------
            if 수치가능:
                유효 = 숫자.dropna()
                감지 = sorted({v for v in SENTINELS if (유효 == v).any()})
                if 감지:
                    개수 = int(유효.isin(감지).sum())
                    p.issues.append(f"결측 위장값 {감지} {개수}건 (실제 결측일 가능성)")
                    p.fixes.append("sentinel_to_nan")

            빈문자 = int(문자.str.strip().str.lower().isin(SENTINEL_STRINGS).sum())
            if 빈문자:
                p.issues.append(f"결측을 뜻하는 문자열 {빈문자}건")
                p.fixes.append("string_to_nan")

            # --- 타입 판정 -----------------------------------------
            if p.n_unique == n and n > 10:
                p.kind = "id"
            elif 수치가능:
                유효 = 숫자[~숫자.isin(SENTINELS)].dropna()
                if len(유효) >= 3:
                    p.stats = {
                        "mean": round(float(유효.mean()), 4),
                        "sd": round(float(유효.std(ddof=1)), 4),
                        "min": round(float(유효.min()), 4),
                        "median": round(float(유효.median()), 4),
                        "max": round(float(유효.max()), 4),
                        "skew": round(float(유효.skew()), 3),
                    }
                    self._check_outliers(p, 유효)
                    self._check_normality(p, 유효)
                # 정수이고 수준이 적으면 순서형일 수 있다
                정수형 = bool((유효 % 1 == 0).all()) if len(유효) else False
                if 정수형 and p.n_unique <= self.ordinal_max_levels:
                    p.kind = "ordinal"
                    p.issues.append(
                        f"정수 {p.n_unique}수준 — 순서형일 수 있음 "
                        f"(리커트 척도 등). 평균을 내는 것이 적절한지 확인할 것"
                    )
                else:
                    p.kind = "continuous"
                if 변환실패:
                    p.issues.append(f"수치로 못 바꾸는 값 {변환실패}건")
            else:
                정규화 = 문자.str.strip().str.lower()
                수준 = sorted(정규화.unique())
                p.levels = [str(v) for v in sorted(문자.unique())][:20]
                if len(수준) < p.n_unique:
                    p.issues.append(
                        f"대소문자·공백만 다른 값이 섞여 있음 "
                        f"(표기 {p.n_unique}종 → 실제 {len(수준)}종)"
                    )
                    p.fixes.append("normalize_case")

                # 축약형과 완전형이 섞였는가 (y / yes)
                family = self._match_synonym_family(수준)
                if family is not None:
                    합친수준 = sorted({family[v] for v in 수준})
                    if len(합친수준) < len(수준):
                        p.issues.append(
                            f"같은 뜻의 다른 표기가 섞여 있음 "
                            f"({수준} → {합친수준})"
                        )
                        p.fixes.append("merge_synonyms")
                        수준 = 합친수준
                p.kind = "binary" if len(수준) == 2 else "nominal"

            # --- 결측률 -------------------------------------------
            결측률 = p.n_missing / n if n else 0
            if 결측률 > 0.3:
                p.issues.append(f"결측률 {결측률:.0%} — 분석에서 제외를 고려")
            elif 결측률 > 0:
                p.issues.append(f"결측 {p.n_missing}건 ({결측률:.1%})")

            self.profiles[col] = p

    # ----------------------------------------------------------------
    @staticmethod
    def _match_synonym_family(levels: list[str]) -> dict[str, str] | None:
        """이 열의 값들이 알려진 동의어 집합에 전부 들어가는가.

        일부만 걸치면 `None` 을 돌려준다. 예를 들어 값이
        `['y', 'n', 'maybe']` 라면 함부로 합치지 않는다 —
        모르는 값이 섞여 있으면 손대지 않는 것이 안전하다.
        """
        for family in SYNONYM_FAMILIES:
            if all(v in family for v in levels):
                return family
        return None

    @staticmethod
    def _check_outliers(p: ColumnProfile, values: Any) -> None:
        q1, q3 = values.quantile([0.25, 0.75])
        iqr = q3 - q1
        if iqr <= 0:
            return
        low, high = q1 - 3 * iqr, q3 + 3 * iqr      # 3배 = 극단값만
        극단 = values[(values < low) | (values > high)]
        if len(극단):
            p.issues.append(
                f"극단 이상치 {len(극단)}건 "
                f"(예: {[round(float(v), 1) for v in 극단.head(3)]}) — 입력 오류 가능"
            )
            p.stats["outlier_bounds"] = [round(float(low), 2), round(float(high), 2)]

    @staticmethod
    def _check_normality(p: ColumnProfile, values: Any) -> None:
        if len(values) < 3:
            return
        try:
            from scipy import stats

            pv = float(stats.shapiro(values[:5000]).pvalue)
            p.stats["shapiro_p"] = pv
            p.stats["normal_at_0.05"] = bool(pv >= 0.05)
        except Exception:
            pass

    # ----------------------------------------------------------------
    def report(self) -> str:
        """사람이 읽는 진단 보고서."""
        줄 = [f"데이터: {len(self.df)}행 {len(self.df.columns)}열", ""]
        for kind in ["id", "continuous", "ordinal", "binary", "nominal", "unknown"]:
            해당 = [p for p in self.profiles.values() if p.kind == kind]
            if not 해당:
                continue
            줄.append(f"[{kind}] {len(해당)}개")
            줄 += [p.line() for p in 해당]
            줄.append("")
        문제열 = [p.name for p in self.profiles.values() if p.fixes]
        if 문제열:
            줄.append(f"정리가 필요한 열: {문제열}")
        return "\n".join(줄)

    def to_prompt(self, max_cols: int = 30) -> str:
        """LLM 프롬프트에 넣을 압축 요약.

        `report()` 보다 짧다. 토큰 예산 때문이다 (4주차).
        """
        줄 = [f"행 {len(self.df)}개, 열 {len(self.df.columns)}개"]
        for p in list(self.profiles.values())[:max_cols]:
            부분 = f"- {p.name}: {p.kind}"
            if p.kind in ("binary", "nominal") and p.levels:
                부분 += f" {p.levels[:6]}"
            if p.stats.get("mean") is not None:
                부분 += f" (평균 {p.stats['mean']}, 결측 {p.n_missing})"
            if p.stats.get("normal_at_0.05") is False:
                부분 += " [정규성 기각]"
            if p.issues:
                부분 += f"  ※ {p.issues[0]}"
            줄.append(부분)
        return "\n".join(줄)

    def issues(self) -> dict[str, list[str]]:
        return {n: p.issues for n, p in self.profiles.items() if p.issues}

    # ----------------------------------------------------------------
    def clean(self, *, drop_outliers: bool = False, verbose: bool = True) -> Any:
        """진단에서 찾은 문제를 실제로 고친 DataFrame 을 돌려준다.

        **원본을 바꾸지 않는다.** 무엇을 고쳤는지 출력한다 —
        조용히 데이터를 바꾸는 것은 통계 분석에서 가장 위험한 일이다.
        """
        import numpy as np
        import pandas as pd

        out = self.df.copy()
        기록 = []

        for name, p in self.profiles.items():
            s = out[name]

            if "comma_decimal" in p.fixes:
                s = s.astype(str).str.replace(
                    r"^(\s*[-+]?\d{1,3}),(\d{1,2}\s*)$", r"\1.\2", regex=True)
                기록.append(f"{name}: 소수점 쉼표 → 마침표")

            if "strip_units" in p.fixes:
                s = s.astype(str).str.replace(r"\s*[a-zA-Z가-힣%°]+\s*$", "", regex=True)
                기록.append(f"{name}: 단위 문자 제거")

            if "string_to_nan" in p.fixes:
                s = s.replace(
                    {v: np.nan for v in SENTINEL_STRINGS}
                ).replace(r"^\s*$", np.nan, regex=True)
                기록.append(f"{name}: 결측 문자열 → NaN")

            if "normalize_case" in p.fixes:
                s = s.astype(str).str.strip().str.lower()
                기록.append(f"{name}: 대소문자·공백 정규화")

            if "merge_synonyms" in p.fixes:
                정규 = s.astype(str).str.strip().str.lower()
                family = self._match_synonym_family(sorted(정규.dropna().unique()))
                if family is not None:
                    전 = sorted(정규.unique())
                    s = 정규.map(family)
                    기록.append(f"{name}: 동의어 통합 {전} → {sorted(s.dropna().unique())}")

            if p.kind in ("continuous", "ordinal"):
                s = pd.to_numeric(s, errors="coerce")
                if "sentinel_to_nan" in p.fixes:
                    before = int(s.notna().sum())
                    s = s.mask(s.isin(SENTINELS))
                    기록.append(f"{name}: 결측 위장값 → NaN ({before - int(s.notna().sum())}건)")

                if drop_outliers and "outlier_bounds" in p.stats:
                    low, high = p.stats["outlier_bounds"]
                    before = int(s.notna().sum())
                    s = s.mask((s < low) | (s > high))
                    지운수 = before - int(s.notna().sum())
                    if 지운수:
                        기록.append(f"{name}: 극단 이상치 {지운수}건 → NaN")

            out[name] = s

        if verbose:
            if 기록:
                print("적용한 정리:")
                for r in 기록:
                    print(f"  - {r}")
            else:
                print("정리할 것이 없습니다.")
            if not drop_outliers:
                이상 = [n for n, p in self.profiles.items() if "outlier_bounds" in p.stats]
                if 이상:
                    print(f"\n[주의] 이상치가 있는 열: {이상}")
                    print("       drop_outliers=True 로 제거할 수 있으나,")
                    print("       제거 여부는 분석자가 판단할 일입니다.")
        return out


# --------------------------------------------------------------------------
# 검정 선택 규칙 — 도메인 지식을 코드로 고정한다
# --------------------------------------------------------------------------
TEST_RULES: dict[str, dict[str, Any]] = {
    "paired-t-test": {
        "설명": "같은 대상의 두 시점 비교, 차이가 정규",
        "outcome": ["continuous"],
        "design": "paired",
        "n_groups": 2,
        "normality_required": True,
    },
    "wilcoxon-signed-rank": {
        "설명": "같은 대상의 두 시점 비교, 정규성 불만족",
        "outcome": ["continuous", "ordinal"],
        "design": "paired",
        "n_groups": 2,
        "normality_required": False,
    },
    "independent-t-test": {
        "설명": "서로 다른 두 집단 비교, 정규",
        "outcome": ["continuous"],
        "design": "independent",
        "n_groups": 2,
        "normality_required": True,
    },
    "mann-whitney-u": {
        "설명": "서로 다른 두 집단 비교, 정규성 불만족",
        "outcome": ["continuous", "ordinal"],
        "design": "independent",
        "n_groups": 2,
        "normality_required": False,
    },
    "one-way-anova": {
        "설명": "세 집단 이상 비교, 정규 + 등분산",
        "outcome": ["continuous"],
        "design": "independent",
        "n_groups": "3+",
        "normality_required": True,
    },
    "kruskal-wallis": {
        "설명": "세 집단 이상 비교, 정규성 불만족",
        "outcome": ["continuous", "ordinal"],
        "design": "independent",
        "n_groups": "3+",
        "normality_required": False,
    },
    "chi-square": {
        "설명": "두 범주형 변수의 연관성",
        "outcome": ["nominal", "binary"],
        "design": "independent",
        "n_groups": "any",
        "normality_required": False,
    },
    "pearson-correlation": {
        "설명": "두 연속형 변수의 선형 관계, 정규",
        "outcome": ["continuous"],
        "design": "correlation",
        "n_groups": None,
        "normality_required": True,
    },
    "spearman-correlation": {
        "설명": "두 변수의 단조 관계, 정규성 불만족 또는 순서형",
        "outcome": ["continuous", "ordinal"],
        "design": "correlation",
        "n_groups": None,
        "normality_required": False,
    },
}


def validate_plan(plan: dict[str, Any], profiler: DataProfiler) -> list[str]:
    """LLM이 세운 분석 계획을 **규칙으로 검증**한다.

    이것이 이 모듈의 핵심이다. LLM은 "이 질문에는 t검정이 맞겠다" 까지만
    하고, 그것이 **데이터의 실제 성질과 맞는지는 코드가 확인**한다.

    계획 형식::

        {
            "test": "independent-t-test",
            "outcome_column": "yield_after",
            "group_column": "fertilizer",     # 선택
            "second_column": "water_liters",  # 상관분석용, 선택
            "design": "independent",
            "reason": "...",
        }
    """
    문제: list[str] = []
    test = plan.get("test")

    if test not in TEST_RULES:
        return [f"'{test}' 는 알 수 없는 검정입니다. 가능: {sorted(TEST_RULES)}"]

    rule = TEST_RULES[test]
    prof = profiler.profiles

    # --- 결과변수 --------------------------------------------------
    outcome = plan.get("outcome_column")
    if not outcome:
        문제.append("outcome_column 이 없습니다.")
    elif outcome not in prof:
        문제.append(f"'{outcome}' 열이 데이터에 없습니다. 있는 열: {list(prof)}")
    else:
        kind = prof[outcome].kind
        if kind not in rule["outcome"]:
            문제.append(
                f"{test} 는 결과변수가 {rule['outcome']} 이어야 하는데 "
                f"'{outcome}' 는 {kind} 입니다."
            )
        if kind == "id":
            문제.append(f"'{outcome}' 는 식별자로 보입니다. 분석 대상이 아닙니다.")

    # --- 집단변수와 집단 수 ----------------------------------------
    # 대응표본은 '집단 열'이 아니라 '두 개의 측정 열'을 쓴다.
    # n_groups=2 라는 표기에 낚여 group_column 을 요구하면 안 된다.
    group = plan.get("group_column")
    집단열_필요 = rule["design"] == "independent" and rule["n_groups"] in (2, "3+")
    if 집단열_필요:
        if not group:
            문제.append(f"{test} 에는 group_column 이 필요합니다.")
        elif group not in prof:
            문제.append(f"'{group}' 열이 데이터에 없습니다.")
        else:
            수준수 = prof[group].n_unique
            if rule["n_groups"] == 2 and 수준수 != 2:
                대안 = "one-way-anova / kruskal-wallis" if 수준수 > 2 else "(수준이 1개)"
                문제.append(
                    f"{test} 는 두 집단 비교인데 '{group}' 의 수준이 {수준수}개입니다. "
                    f"→ {대안} 를 고려하세요."
                )
            if rule["n_groups"] == "3+" and 수준수 < 3:
                문제.append(
                    f"{test} 는 세 집단 이상인데 '{group}' 의 수준이 {수준수}개입니다. "
                    f"→ independent-t-test / mann-whitney-u 를 고려하세요."
                )
            if prof[group].kind == "continuous":
                문제.append(
                    f"'{group}' 는 연속형입니다. 집단 변수로 쓰려면 범주화가 필요합니다."
                )

    # --- 대응 설계 --------------------------------------------------
    design = plan.get("design")
    if design and design != rule["design"]:
        문제.append(
            f"{test} 는 {rule['design']} 설계용인데 계획에는 {design} 이라고 되어 있습니다."
        )
    if rule["design"] == "paired" and not plan.get("second_column"):
        문제.append(
            "대응표본 검정에는 두 번째 측정 열(second_column)이 필요합니다."
        )

    # --- 상관분석 ---------------------------------------------------
    if rule["design"] == "correlation":
        second = plan.get("second_column")
        if not second:
            문제.append("상관분석에는 second_column 이 필요합니다.")
        elif second not in prof:
            문제.append(f"'{second}' 열이 데이터에 없습니다.")
        elif prof[second].kind not in rule["outcome"]:
            문제.append(
                f"{test} 는 두 변수가 모두 {rule['outcome']} 이어야 하는데 "
                f"'{second}' 는 {prof[second].kind} 입니다."
            )

    # --- 정규성 -----------------------------------------------------
    if rule["normality_required"] and outcome in prof:
        정규 = prof[outcome].stats.get("normal_at_0.05")
        if 정규 is False:
            비모수 = {
                "paired-t-test": "wilcoxon-signed-rank",
                "independent-t-test": "mann-whitney-u",
                "one-way-anova": "kruskal-wallis",
                "pearson-correlation": "spearman-correlation",
            }.get(test, "비모수 검정")
            문제.append(
                f"'{outcome}' 는 정규성 검정에서 기각되었습니다 "
                f"(Shapiro p={prof[outcome].stats.get('shapiro_p', 0):.2e}). "
                f"→ {비모수} 를 고려하세요."
            )

    return 문제


# --------------------------------------------------------------------------
# 분석계획 수립기
# --------------------------------------------------------------------------
PLANNER_SYSTEM = """\
너는 통계 분석 계획을 세우는 도구다. 데이터 요약과 사용자 질문을 받아
어떤 검정을 어떤 변수로 수행할지 정한다.

너는 계산하지 않는다. 계획만 세운다.
데이터에 실제로 있는 열 이름만 쓴다. 없는 열을 지어내지 않는다.
확신이 없으면 reason 에 그렇게 적는다."""


class AnalysisPlanner:
    """LLM이 계획을 세우고, 규칙이 검증하고, 실패하면 되먹인다.

    3주차 `ask_json()` 과 같은 구조다. 달라진 것은 검증이
    **스키마 검사가 아니라 통계적 타당성 검사**라는 점이다.
    """

    def __init__(self, llm: Any, profiler: DataProfiler, *, n_retry: int = 3,
                 verbose: bool = True) -> None:
        self.llm = llm
        self.profiler = profiler
        self.n_retry = n_retry
        self.verbose = verbose

    def plan(self, question: str, **kw: Any) -> dict[str, Any]:
        """질문에 대한 분석 계획을 세운다.

        Returns
        -------
        dict : 검증을 통과한 계획.
               실패하면 {"test": None, "error": ..., "attempts": [...]}
        """
        from structured import JSONParseError, parse_json

        schema_desc = {
            "test": f"다음 중 하나: {sorted(TEST_RULES)}",
            "outcome_column": "분석할 결과변수 열 이름",
            "group_column": "집단 비교면 집단 열 이름, 아니면 null",
            "second_column": "대응표본의 두 번째 측정 열 또는 상관분석의 두 번째 변수, 아니면 null",
            "design": "paired / independent / correlation 중 하나",
            "reason": "이 검정을 고른 이유 한두 문장",
        }
        규칙설명 = "\n".join(
            f"- {k}: {v['설명']} (결과변수 {v['outcome']}, 설계 {v['design']}, 집단 {v['n_groups']})"
            for k, v in TEST_RULES.items()
        )

        base = (
            f"## 데이터 요약\n{self.profiler.to_prompt()}\n\n"
            f"## 선택 가능한 검정\n{규칙설명}\n\n"
            f"## 질문\n{question}\n\n"
            f"## 출력 형식 (JSON만)\n"
            + "\n".join(f'  "{k}": {v}' for k, v in schema_desc.items())
            + "\n\n설명이나 코드블록 없이 JSON 객체 하나만 출력하라."
        )

        messages = [
            {"role": "system", "content": PLANNER_SYSTEM},
            {"role": "user", "content": base},
        ]
        kw.setdefault("temperature", 0.0)
        kw.setdefault("max_tokens", 500)
        시도들 = []

        for attempt in range(1, self.n_retry + 1):
            try:
                원문 = self.llm.chat(messages, **kw)
            except Exception as exc:
                # 서버 문제로 루프 전체가 죽으면 안 된다 (6주차와 같은 원칙)
                시도들.append({"attempt": attempt, "error": f"{type(exc).__name__}: {exc}"})
                if self.verbose:
                    print(f"[{attempt}회] 모델 호출 실패: {type(exc).__name__}")
                if attempt == self.n_retry:
                    return {"test": None,
                            "error": f"모델 호출 실패: {type(exc).__name__}: {exc}",
                            "attempts": 시도들}
                continue

            try:
                계획 = parse_json(원문)
            except JSONParseError as exc:
                시도들.append({"attempt": attempt, "error": str(exc)})
                피드백 = f"JSON으로 읽을 수 없었습니다: {exc}\nJSON 객체만 출력하세요."
            else:
                문제 = validate_plan(계획, self.profiler)
                시도들.append({"attempt": attempt, "plan": 계획, "problems": 문제})
                if not 문제:
                    if self.verbose:
                        print(f"[계획 수립 성공 - {attempt}회] {계획.get('test')}")
                    계획["attempts"] = 시도들
                    return 계획
                if self.verbose:
                    print(f"[{attempt}회] {계획.get('test')} → 검증 실패: {문제[0]}")
                피드백 = (
                    "이 계획은 데이터의 성질과 맞지 않습니다:\n"
                    + "\n".join(f"- {p}" for p in 문제)
                    + "\n위 지적을 반영해 계획을 고쳐 JSON만 다시 출력하세요."
                )

            messages.append({"role": "assistant", "content": 원문})
            messages.append({"role": "user", "content": 피드백})

        return {
            "test": None,
            "error": f"{self.n_retry}회 시도 모두 검증을 통과하지 못했습니다.",
            "attempts": 시도들,
        }


In [ ]:
%%writefile memory.py
"""전산실험2 4주차 : 대화 상태와 메모리.

모델은 이전 대화를 기억하지 못한다. 기억은 **우리 코드가** 만든다.
그런데 대화를 통째로 다시 보내다 보면 컨텍스트 윈도우에 부딪힌다.

이 모듈은 세 가지를 제공한다.

    Conversation   대화 기록을 들고 있는 그릇 (system 은 항상 보존)
    예산 관리       토큰을 세고, 넘치면 오래된 턴을 잘라낸다
    요약 압축       잘라내는 대신 요약으로 갈아끼운다 (정보 손실 최소화)

사용 예:
    from kmu_llm import LLM
    from memory import Conversation

    llm = LLM()
    conv = Conversation(llm, system="너는 통계 조교다.", max_tokens=2000)
    print(conv.say("표준편차가 뭔가요?"))
    print(conv.say("그럼 표준오차는요?"))     # 앞 대화를 기억한다
    conv.summary()                          # 현재 토큰 사용 현황
"""

from __future__ import annotations

from typing import Any, Literal

__all__ = ["estimate_tokens", "Message", "Conversation"]

Role = Literal["system", "user", "assistant"]


# --------------------------------------------------------------------------
# 토큰 추정
# --------------------------------------------------------------------------
def estimate_tokens(text: str) -> int:
    """토큰 수를 대략 추정한다. 서버에 묻지 않고 빠르게 계산한다.

    정확한 값은 모델의 토크나이저에 따라 다르다. 여기서는 경험적 근사를 쓴다.

        한글 1글자  ~ 1.2 토큰   (자소 단위로 잘게 쪼개짐)
        영문 1글자  ~ 0.3 토큰   (단어가 통째로 1토큰이 되는 경우가 많음)

    루프를 돌 때마다 서버에 물어보면 호출이 배로 늘어난다. 예산 관리는
    **근사로 충분**하고, 정확한 값이 필요하면 `LLM.count_tokens()` 를 쓴다.
    """
    if not text:
        return 0
    hangul = sum(1 for ch in text if "가" <= ch <= "힣")
    other = len(text) - hangul
    return int(hangul * 1.2 + other * 0.3) + 1


class Message(dict):
    """{"role": ..., "content": ...} 에 토큰 수를 얹은 것."""

    def __init__(self, role: Role, content: str) -> None:
        super().__init__(role=role, content=content)

    @property
    def tokens(self) -> int:
        return estimate_tokens(self["content"]) + 4  # 역할 표시 등 부대 비용

    def as_api(self) -> dict[str, str]:
        return {"role": self["role"], "content": self["content"]}


# --------------------------------------------------------------------------
# 대화
# --------------------------------------------------------------------------
class Conversation:
    """대화 기록을 관리하고 예산을 넘지 않게 유지한다.

    Parameters
    ----------
    llm : LLM 인스턴스
    system : 대화 전체에 적용되는 지시. **절대 잘려나가지 않는다.**
    max_tokens : 전체 대화의 토큰 예산 (응답 공간은 별도)
    strategy :
        "drop"    오래된 턴을 그냥 버린다 (빠르고 단순, 정보 손실)
        "summary" 오래된 턴을 요약으로 갈아끼운다 (호출 1회 추가, 손실 적음)
        "none"    아무것도 하지 않는다 (한계를 직접 보고 싶을 때)
    keep_recent : 최근 몇 개의 메시지는 무조건 원본으로 보존할지
    """

    def __init__(
        self,
        llm: Any,
        system: str | None = None,
        *,
        max_tokens: int = 3000,
        strategy: str = "summary",
        keep_recent: int = 4,
    ) -> None:
        if strategy not in ("drop", "summary", "none"):
            raise ValueError(f"알 수 없는 strategy: {strategy!r}")

        self.llm = llm
        self.max_tokens = max_tokens
        self.strategy = strategy
        self.keep_recent = keep_recent

        self.system = Message("system", system) if system else None
        self.history: list[Message] = []
        self.compressions = 0          # 몇 번 압축했는지
        self.dropped = 0               # 몇 개를 버렸는지

    # -- 상태 확인 --------------------------------------------------------
    def messages(self) -> list[dict[str, str]]:
        """API 로 보낼 형태."""
        head = [self.system] if self.system else []
        return [m.as_api() for m in head + self.history]

    @property
    def tokens(self) -> int:
        head = self.system.tokens if self.system else 0
        return head + sum(m.tokens for m in self.history)

    def summary(self) -> dict[str, Any]:
        """현재 상태를 한눈에."""
        return {
            "메시지수": len(self.history) + (1 if self.system else 0),
            "추정토큰": self.tokens,
            "예산": self.max_tokens,
            "사용률": round(self.tokens / self.max_tokens, 3),
            "압축횟수": self.compressions,
            "버린메시지": self.dropped,
        }

    # -- 대화 -------------------------------------------------------------
    def add(self, role: Role, content: str) -> None:
        self.history.append(Message(role, content))

    def say(self, text: str, **kw: Any) -> str:
        """사용자 발화를 넣고 모델의 답을 받아 기록에 남긴다.

        예산은 **보내기 전과 받은 뒤 두 번** 맞춘다. 그래야
        "대화는 항상 예산 안에 있다"는 불변식이 유지되어,
        지금 상태를 보고 다음 호출이 들어갈지 예측할 수 있다.
        """
        self.add("user", text)
        self.fit()                                  # 보내기 전
        reply = self.llm.chat(self.messages(), **kw)
        self.add("assistant", reply)
        self.fit()                                  # 받은 뒤
        return reply

    # -- 예산 관리 --------------------------------------------------------
    def fit(self) -> None:
        """예산을 넘으면 전략에 따라 줄인다."""
        if self.strategy == "none" or self.tokens <= self.max_tokens:
            return
        if self.strategy == "drop":
            self._drop_old()
        else:
            self._summarize_old()

    def _split(self) -> tuple[list[Message], list[Message]]:
        """(압축 대상인 오래된 것, 보존할 최근 것)"""
        if len(self.history) <= self.keep_recent:
            return [], self.history
        return self.history[: -self.keep_recent], self.history[-self.keep_recent :]

    def _drop_old(self) -> None:
        """가장 오래된 것부터 예산에 맞을 때까지 버린다.

        `keep_recent` 는 보존하려 하지만, 그것만으로도 예산을 넘으면
        보존 약속을 깨고 더 버린다. 마지막 메시지 하나는 남긴다 —
        그것이 사용자의 방금 질문이기 때문이다.
        """
        old, recent = self._split()
        while old and self.tokens > self.max_tokens:
            old.pop(0)
            self.dropped += 1
            self.history = old + recent
        self.history = old + recent

        # 최근 것만 남았는데도 넘치면, 보존 약속을 깨고 더 버린다
        while len(self.history) > 1 and self.tokens > self.max_tokens:
            self.history.pop(0)
            self.dropped += 1

        if self.tokens > self.max_tokens:
            # 마지막 메시지 하나가 예산보다 크다. 내용을 몰래 자르지는 않는다.
            # 조용히 틀리느니 시끄럽게 알리는 편이 낫다. (3주차와 같은 원칙)
            print(
                f"[경고] 마지막 메시지 하나가 예산({self.max_tokens} 토큰)을 "
                f"넘습니다 (현재 {self.tokens}). 예산을 늘리거나 입력을 줄이세요."
            )

    def _summarize_old(self) -> None:
        """오래된 대화를 요약 한 덩어리로 갈아끼운다."""
        old, recent = self._split()
        if not old:
            self._drop_old()          # 줄일 것이 최근 것뿐이면 버리는 수밖에 없다
            return

        기록 = "\n".join(f"[{m['role']}] {m['content']}" for m in old)
        지시 = (
            "다음은 사용자와 조수의 지난 대화다. 이후 대화를 이어가는 데 필요한 "
            "사실만 5문장 이내로 요약하라. 결정된 내용, 사용자가 밝힌 정보, "
            "아직 해결되지 않은 질문을 우선 남겨라. 인사말은 버려라.\n\n"
            f"{기록}"
        )
        try:
            요약 = self.llm.ask(지시, temperature=0.0, max_tokens=400)
        except Exception:
            self._drop_old()          # 요약 호출이 실패하면 버리기로 대체
            return

        self.history = [Message("user", f"[이전 대화 요약]\n{요약}")] + recent
        self.compressions += 1

        if self.tokens > self.max_tokens:   # 요약해도 넘치면 결국 버린다
            self._drop_old()

    # -- 저장 / 복원 ------------------------------------------------------
    def to_dict(self) -> dict[str, Any]:
        return {
            "system": self.system["content"] if self.system else None,
            "history": [m.as_api() for m in self.history],
            "compressions": self.compressions,
            "dropped": self.dropped,
        }

    @classmethod
    def from_dict(cls, llm: Any, data: dict[str, Any], **kw: Any) -> "Conversation":
        conv = cls(llm, system=data.get("system"), **kw)
        for m in data.get("history", []):
            conv.add(m["role"], m["content"])
        conv.compressions = data.get("compressions", 0)
        conv.dropped = data.get("dropped", 0)
        return conv


In [ ]:
!pip install -q openai

# 이 수업 저장소에서 공용 모듈 내려받기

from kmu_llm import LLM, ask, check_server

PROVIDER = "kmu"          # 학과 서버가 안 되면 "gemini" 로 바꾸세요
llm = LLM(provider=PROVIDER)
print("준비 완료:", llm)

In [ ]:
%%writefile exam_greenhouse.csv
pot_id,fertilizer,greenhouse,yield_before,yield_after,water_liters,temp_c,leaf_count,pest_damage
G0001,C,south,63.63,63.91,2.33,25.8,30.0,No
G0002,C,north,44.55,38.06,2.23,22.1,22.0,no
G0003,A,north,28.47,28.83,2.85,20.5,30.0,no
G0004,B,south,60.01,58.01,0.78,-999,34.0,No
G0005,B,south,68.94,85.16,1.57,25.4,34.0,No
G0006,A,south,41.56,48.18,"2,58",25.1,32.0,no
G0007,A,south,48.84,48.73,2.5,25.0,47.0,No
G0008,A,south,56.79,56.91,1.73,21.4,31.0,No
G0009,A,south,39.05,42.84,2.04,22.7,33.0,No
G0010,A,south,58.79,60.51,2.12,-999,35.0,Yes
G0011,B,north,61.89,71.84,2.01,20.3,27.0,No
G0012,A,north,53.67,61.11,2.89,23.7,32.0,Yes
G0013,A,south,52.87,57.02,3.55,24.6,29.0,Y
G0014,C,north,49.44,43.73,1.72,22.0,49.0,no
G0015,C,south,44.32,51.82,2.98,25.0,36.0,no
G0016,C,north,43.19,41.9,2.32,23.0,26.0,no
G0017,A,south,56.04,56.93,1.97,27.0,28.0,no
G0018,A,south,58.25,68.38,2.25,22.2,22.0,no
G0019,B,south,66.43,81.83,2.32,27.6,33.0,No
G0020,B,north,512.0,67.2,2.56,29.7,27.0,No
G0021,A,south,44.29,42.75,3.57,27.7,32.0,No
G0022,A,north,34.38,34.04,2.06,20.3,24.0,no
G0023,A,south,55.24,62.74,2.59,20.5,33.0,No
G0024,C,north,53.45,60.79,3.19,22.4,26.0,No
G0025,A,south,55.04,61.16,2.89,21.4,26.0,no
G0026,A,south,66.51,66.72,2.26,28.2,36.0,no
G0027,C,north,43.58,50.97,2.79,26.9,26.0,No
G0028,B,south,59.41,67.56,2.62,16.8,30.0,no
G0029,B,north,63.05,69.99,2.97,22.7,36.0,No
G0030,B,north,53.61,60.34,3.61,20.5,37.0,no
G0031,A,north,49.31,50.56,1.38,27.2,34.0,No
G0032,B,south,51.58,61.88,2.52,-999,27.0,No
G0033,B,south,52.52,59.53,2.43,23.6,,No
G0034,B,south,64.49,68.34,2.16,24.9,30.0,no
G0035,C,north,44.66,51.11,2.95,19.3,35.0,no
G0036,C,south,54.42,63.11,3.35,26.2,31.0,No
G0037,C,south,46.66,53.42,1.82,-999,30.0,No
G0038,A,north,47.05,47.01,2.56,22.2,24.0,no
G0039,B,south,69.03,73.57kg,2.6,22.5,19.0,no
G0040,A,south,52.02,56.99,2.2,24.5,34.0,no
G0041,B,north,73.02,83.96,2.35,-999,,No
G0042,B,north,57.19,67.53,2.21,22.3,38.0,No
G0043,C,south,48.89,51.07,3.31,28.4,22.0,no
G0044,C,south,45.51,43.91,2.03,25.5,35.0,No
G0045,C,south,73.1,72.83,2.67,22.4,36.0,no
G0046,A,south,49.77,53.55,2.16,21.1,26.0,no
G0047,A,south,50.02,54.03,1.98,20.2,25.0,no
G0048,B,north,52.5,60.99,2.18,25.8,22.0,No
G0049,C,south,47.19,48.26,2.22,27.9,32.0,No
G0050,C,south,52.92,57.25,"2,28",25.5,39.0,yes
G0051,A,south,52.14,50.21,3.8,24.8,27.0,No
G0052,C,north,58.18,53.26,3.48,21.7,24.0,No
G0053,B,north,60.12,72.63,1.81,21.3,38.0,No
G0054,C,south,48.38,45.24,2.59,-999,17.0,No
G0055,A,south,68.79,71.0,2.87,25.7,27.0,no
G0056,C,south,55.3,54.23,1.96,22.3,40.0,No
G0057,B,north,54.27,59.11,2.5,20.2,30.0,no
G0058,A,north,51.87,51.74,2.79,23.9,39.0,no
G0059,B,south,57.39,72.06,2.1,24.4,26.0,no
G0060,A,north,55.27,59.75,2.22,27.7,31.0,No
G0061,C,south,40.18,46.98,1.55,22.6,34.0,Y
G0062,A,north,42.66,49.74,1.95,25.9,34.0,yes
G0063,C,south,43.26,41.39,2.6,24.1,27.0,No
G0064,A,south,48.12,51.89,1.6,24.6,34.0,No
G0065,B,north,52.48,52.5,1.57,23.5,31.0,No
G0066,A,north,42.64,42.22,2.19,21.4,28.0,No
G0067,B,south,59.28,63.65,"1,91",22.8,37.0,no
G0068,A,north,49.49,51.99,2.58,23.4,40.0,No
G0069,B,south,60.73,65.78,2.81,19.2,31.0,No
G0070,C,north,58.42,63.93,2.28,27.5,30.0,No
G0071,A,south,56.53,55.65,2.68,19.5,40.0,No
G0072,A,north,42.11,44.37,2.54,24.3,31.0,No
G0073,A,north,52.74,51.42,1.51,24.4,23.0,yes
G0074,C,north,50.85,54.03,2.31,25.1,42.0,No
G0075,C,south,47.49,54.63,1.91,27.1,31.0,no
G0076,A,north,43.56,47.87,3.95,22.5,35.0,Yes
G0077,C,south,52.88,49.8,3.15,27.3,37.0,Yes
G0078,B,south,67.53,72.61,2.61,23.4,31.0,yes
G0079,B,north,56.41,60.02,2.77,22.3,31.0,no
G0080,B,north,62.92,58.97,3.46,26.3,28.0,No
G0081,B,south,63.73,66.0,2.28,27.9,36.0,No
G0082,C,south,45.18,46.22,1.4,25.0,28.0,No
G0083,B,south,59.25,67.01,2.2,24.6,30.0,No
G0084,C,south,59.92,67.33,1.85,18.8,30.0,Y
G0085,A,north,52.66,53.71,1.62,25.0,,no
G0086,C,south,47.7,42.38,2.9,28.3,26.0,no
G0087,A,south,60.8,66.25,2.82,27.7,29.0,No
G0088,B,north,68.3,74.46,2.34,27.4,24.0,No
G0089,A,north,41.89,47.62,2.21,24.2,30.0,Yes
G0090,C,north,52.73,55.13,2.61,28.4,29.0,no
G0091,A,north,52.56,56.19,2.46,25.1,35.0,Yes
G0092,B,north,57.11,58.33,1.96,21.5,40.0,yes
G0093,A,north,44.24,47.81,2.67,18.9,28.0,no
G0094,A,south,55.45,57.43,2.66,26.2,27.0,no
G0095,B,north,58.35,64.22,2.01,18.6,44.0,yes
G0096,B,south,59.22,72.91,1.69,23.3,32.0,No
G0097,B,south,56.92,59.72,2.48,25.0,31.0,No
G0098,B,south,71.57,79.2,"3,06",25.6,28.0,No
G0099,C,south,65.59,63.09,2.72,26.9,36.0,yes
G0100,C,south,55.92,53.6,1.83,25.1,37.0,No
G0101,B,north,56.99,55.43,0.61,23.4,28.0,No
G0102,C,south,55.58,53.03,1.95,26.1,21.0,no
G0103,B,south,57.01,69.18,"2,53",23.2,35.0,yes
G0104,B,north,53.63,60.25,1.88,28.0,27.0,No
G0105,C,south,49.91,51.07,2.38,20.6,26.0,No
G0106,C,south,42.83,43.04,1.89,26.9,35.0,yes
G0107,A,north,52.26,47.42,2.62,23.5,29.0,no
G0108,C,north,47.88,49.17,1.72,23.9,36.0,No
G0109,A,south,53.71,55.9,2.51,25.0,31.0,Y
G0110,C,north,46.1,48.32,1.77,25.6,26.0,No
G0111,B,south,57.65,64.51,2.09,23.8,35.0,No
G0112,C,north,54.48,53.27,2.74,25.1,28.0,No
G0113,A,north,54.8,53.37,2.84,23.8,25.0,no
G0114,B,south,61.46,64.15,2.2,24.7,23.0,No
G0115,C,north,37.61,42.91,2.76,24.9,28.0,No
G0116,A,north,47.21,52.37,2.33,30.1,34.0,no
G0117,B,north,56.41,61.96,"3,56",26.9,33.0,Yes
G0118,B,south,64.33,60.61,2.11,22.5,22.0,No
G0119,B,north,55.62,65.12,1.32,26.7,33.0,No
G0120,A,south,56.54,56.58,1.44,21.1,,no
G0121,A,south,55.05,54.88,3.2,21.2,22.0,no
G0122,A,north,49.31,56.34,2.88,26.8,27.0,no
G0123,C,south,61.84,66.2,2.34,25.2,41.0,No
G0124,B,north,48.53,56.65kg,2.04,25.2,25.0,No
G0125,A,south,37.91,39.22,2.66,29.2,30.0,no
G0126,B,north,59.34,65.72,2.84,23.7,33.0,Yes
G0127,B,north,68.35,74.41,2.0,23.3,36.0,No
G0128,A,north,57.22,48.59,1.38,23.7,24.0,no
G0129,A,south,51.69,49.64,3.01,21.3,32.0,No
G0130,C,south,60.8,62.09,1.73,31.1,29.0,No
G0131,B,south,57.66,60.29,2.77,26.8,32.0,no
G0132,A,north,46.24,52.36,2.73,22.2,33.0,yes
G0133,B,north,57.4,55.62,2.07,27.8,27.0,no
G0134,C,south,36.04,45.16,2.53,22.4,32.0,No
G0135,B,south,51.75,58.96,2.59,21.2,30.0,No
G0136,C,south,59.84,58.73,3.11,23.7,30.0,no
G0137,A,south,66.09,60.19,0.98,22.7,33.0,No
G0138,B,south,59.66,63.69,3.35,21.6,41.0,No
G0139,B,south,60.55,64.43,2.05,28.1,22.0,yes
G0140,B,north,68.88,62.12,2.17,17.6,30.0,Yes
G0141,B,south,72.84,78.61,2.87,26.1,36.0,no
G0142,C,north,49.3,52.61,2.61,23.7,40.0,No
G0143,C,north,50.82,55.4,2.04,24.9,28.0,no
G0144,A,south,59.18,61.6,2.45,26.9,32.0,No
G0145,A,south,53.25,64.61,1.31,28.1,900.0,yes
G0146,A,south,52.95,53.67,3.14,21.1,32.0,No
G0147,A,south,50.76,50.9,2.59,27.2,26.0,No
G0148,A,south,64.77,65.06,2.83,23.0,34.0,no
G0149,A,north,45.39,50.96kg,2.27,20.3,34.0,yes
G0150,C,south,46.53,48.82,2.4,23.2,,yes
G0151,A,south,54.04,58.03,1.67,26.2,33.0,No
G0152,B,north,52.18,60.16kg,2.4,21.5,39.0,no
G0153,A,north,40.83,38.31,2.83,19.5,41.0,no
G0154,B,south,64.8,72.95,1.64,24.7,40.0,Yes
G0155,B,north,35.3,54.17,2.3,25.9,34.0,No
G0156,C,north,49.58,46.72,2.66,21.5,25.0,No
G0157,C,south,48.37,51.85,2.61,18.9,25.0,Yes
G0158,A,north,45.83,43.47,2.56,23.4,34.0,Yes
G0159,C,south,41.96,47.06,2.46,27.3,25.0,Yes
G0160,B,north,59.19,63.6,2.72,24.4,34.0,No
G0161,B,south,55.91,67.17,1.38,23.0,30.0,No
G0162,A,north,56.61,60.92,2.78,23.6,29.0,no
G0163,B,north,50.27,57.73,"2,48",24.1,29.0,no
G0164,C,south,59.15,61.58,2.17,24.4,42.0,yes
G0165,B,north,50.53,51.35,2.85,26.0,32.0,No
G0166,B,north,56.1,65.7,2.72,22.1,29.0,no
G0167,A,north,51.51,45.5,1.63,21.5,29.0,no
G0168,A,north,39.29,39.65kg,2.26,-999,26.0,yes
G0169,A,south,45.81,49.61,1.42,17.4,,Yes
G0170,A,south,51.33,60.67,2.61,23.6,25.0,No
G0171,B,north,58.03,65.13,1.56,26.8,33.0,no
G0172,A,south,47.5,41.3,3.7,27.3,28.0,Yes
G0173,C,south,62.38,55.42,1.75,27.3,24.0,no
G0174,A,north,49.02,51.32,2.11,21.8,34.0,No
G0175,A,south,52.3,61.08,1.97,25.7,37.0,No
G0176,B,south,57.87,67.41,"2,91",25.1,22.0,Y
G0177,C,south,68.87,70.77,2.03,28.3,,No
G0178,A,south,57.68,64.22,2.49,29.2,27.0,Yes
G0179,A,north,39.51,44.83,2.64,23.0,27.0,No
G0180,A,north,38.79,37.13,2.65,28.7,39.0,No
G0181,A,north,70.14,65.83kg,2.45,20.8,33.0,No
G0182,C,north,57.1,59.82,1.83,22.4,26.0,No
G0183,B,north,60.78,66.46,2.55,25.1,36.0,Yes
G0184,B,south,67.57,74.94,2.88,-999,37.0,No
G0185,B,north,51.63,51.22,2.99,26.1,37.0,Yes
G0186,C,north,51.58,51.87,2.31,22.7,39.0,No
G0187,A,north,43.53,43.3,2.58,28.3,19.0,No
G0188,B,north,59.97,58.43,3.95,22.9,30.0,No
G0189,A,north,54.2,55.23,2.46,26.9,29.0,No
G0190,C,south,44.75,47.68,2.29,22.7,31.0,No
G0191,B,south,50.15,59.47,2.15,21.5,34.0,no
G0192,A,south,51.78,64.11,2.41,24.5,36.0,no
G0193,A,south,52.26,50.29,2.39,27.6,24.0,No
G0194,B,south,50.26,58.03,2.07,25.5,32.0,No
G0195,B,north,48.43,50.39,2.28,-999,34.0,No
G0196,C,south,47.54,53.9,2.97,26.1,41.0,No
G0197,B,south,58.36,71.86,2.7,-999,32.0,No
G0198,B,north,65.57,71.83,3.17,23.4,18.0,no
G0199,A,south,53.28,54.1,2.63,16.6,23.0,yes
G0200,A,south,45.83,51.69,1.74,24.5,30.0,No
G0201,A,south,50.09,50.18,1.9,23.1,41.0,No
G0202,B,north,63.8,70.25,2.58,-999,31.0,no
G0203,C,south,63.12,57.35,2.77,21.9,33.0,yes
G0204,B,south,57.3,61.12,3.19,24.8,30.0,No
G0205,C,north,38.13,42.29,0.9,18.6,34.0,no
G0206,C,south,51.98,53.82,3.19,25.4,27.0,No
G0207,A,south,52.94,57.02,2.41,25.7,29.0,No
G0208,B,north,56.92,65.96,1.89,25.9,41.0,No
G0209,C,south,65.93,68.41,3.13,20.5,24.0,No
G0210,B,south,57.97,64.46,2.71,26.8,26.0,No


In [ ]:
%%writefile sample_clinic.csv
patient_id,age,sex,group,bp_before,bp_after,bmi,smoker,satisfaction,income,region
P0001,40.0,M,control,153.9,156.7,25.1,Y,3,12756.0,Busan
P0002,39.0,F,control,146.9,154.0,24.9,N,3,21248.0,Daegu
P0003,49.0,M,treatment,149.4,121.3,31.5,N,4,24254.0,Daegu
P0004,69.0,F,treatment,161.9,155.9,25.7,N,3,56575.0,Daegu
P0005,64.0,M,treatment,168.8,155.2,23.3,N,3,72272.0,Seoul
P0006,48.0,F,treatment,159.8,142.8,25.8,N,4,38254.0,Daegu
P0007,48.0,M,treatment,167.4,156.9,26.9,N,3,7703.0,Daegu
P0008,67.0,M,treatment,480.0,133.9,22.2,N,2,22419.0,Gwangju
P0009,41.0,M,treatment,134.8,111.0,29.2,N,4,22793.0,Seoul
P0010,63.0,F,control,91.7,82.4,24.5,N,2,44750.0,Daegu
P0011,56.0,F,control,143.2,140.8,27.8,N,5,37430.0,Seoul
P0012,53.0,M,control,133.5,129.2,23.7,Y,3,33852.0,Seoul
P0013,61.0,M,treatment,121.7,109.7,29.1,N,4,41307.0,Daegu
P0014,37.0,F,treatment,160.1,143.6,24.4,N,1,53924.0,Daegu
P0015,48.0,F,control,155.3,161.4,30.8,Y,1,92024.0,Gwangju
P0016,30.0,M,control,141.8,137.4,25.4,Y,3,20452.0,Daegu
P0017,70.0,F,control,112.4,111.2,25.2,N,4,23648.0,Seoul
P0018,57.0,M,control,137.3,135.3,20.6,Y,2,10089.0,Seoul
P0019,46.0,F,treatment,123.7,125.5,24.5,N,2,22951.0,Busan
P0020,73.0,F,treatment,162.2,,23.7,Y,3,39878.0,Busan
P0021,57.0,M,control,144.5,134.2,29.9,N,4,75988.0,Busan
P0022,63.0,F,treatment,164.4,157.8,26.0,N,4,65927.0,Busan
P0023,33.0,M,treatment,115.2,109.3,21.5,N,3,40416.0,Busan
P0024,48.0,M,treatment,125.3,108.5,30.6,N,3,19877.0,Gwangju
P0025,44.0,F,control,156.0,161.7,22.5,N,4,44856.0,Seoul
P0026,68.0,M,treatment,156.3,155.5,23.2,Y,1,26890.0,Seoul
P0027,63.0,M,treatment,135.4,111.0,,N,3,7287.0,Daegu
P0028,59.0,F,treatment,132.9,117.3,,N,2,16354.0,Daegu
P0029,43.0,M,control,152.7,156.8,23.0,N,4,26488.0,Daegu
P0030,69.0,F,control,144.4,133.7,24.8,N,2,10399.0,Gwangju
P0031,35.0,F,control,138.2,132.3,27.7,Y,4,10883.0,Busan
P0032,43.0,F,treatment,135.8,117.1,31.1,N,1,36864.0,Busan
P0033,50.0,F,control,156.0,143.6,28.7,N,1,24430.0,Seoul
P0034,50.0,F,treatment,158.9,146.9,23.8,N,3,22788.0,Daegu
P0035,65.0,F,control,162.9,156.7,29.6,N,1,10734.0,Daegu
P0036,54.0,F,treatment,162.2,143.5,27.4,N,2,136149.0,Busan
P0037,69.0,F,control,133.8,127.7,28.6,N,2,50707.0,Seoul
P0038,37.0,M,control,142.0,149.3,23.9,Y,3,18634.0,Daegu
P0039,52.0,M,treatment,165.0,148.7,26.0,N,1,54997.0,Gwangju
P0040,68.0,F,treatment,132.9,114.7,23.6,N,2,17947.0,Daegu
P0041,41.0,F,control,152.3,146.6,23.3,N,1,42218.0,Seoul
P0042,48.0,F,treatment,152.7,153.0,28.3,N,3,11540.0,Daegu
P0043,58세,F,control,138.4,142.3,27.9,N,1,33251.0,Seoul
P0044,68.0,F,treatment,149.0,122.4,32.2,N,1,57593.0,Daegu
P0045,63.0,F,treatment,160.6,152.0,24.2,N,5,36447.0,Busan
P0046,35.0,M,control,172.7,177.5,28.0,Y,4,12268.0,Gwangju
P0047,45.0,F,treatment,141.6,121.4,21.0,Y,5,100463.0,Daegu
P0048,64.0,F,treatment,142.5,117.9,24.1,N,5,21024.0,Daegu
P0049,62.0,M,treatment,151.6,136.5,29.1,N,4,16482.0,Seoul
P0050,88.0,M,control,126.8,112.0,32.0,Y,2,46835.0,Seoul
P0051,62.0,F,control,147.8,146.1,30.9,Y,5,19540.0,Seoul
P0052,51.0,F,treatment,127.1,105.1,28.3,Y,5,60886.0,Daegu
P0053,61.0,F,control,156.6,,31.1,Y,3,22335.0,Seoul
P0054,63.0,M,control,142.9,137.9,23.4,N,4,21115.0,Gwangju
P0055,44.0,F,control,163.1,165.8,26.2,N,1,17212.0,Daegu
P0056,50.0,F,control,134.5,131.7,30.4,N,2,48016.0,Daegu
P0057,46.0,F,treatment,148.8,134.7,26.7,N,2,5933.0,Seoul
P0058,43.0,M,treatment,149.4,127.6,25.2,N,3,55878.0,Busan
P0059,69.0,F,control,145.2,136.6,24.0,N,5,22919.0,Seoul
P0060,39.0,F,control,159.1,153.1,30.4,Y,3,38223.0,Daegu
P0061,52.0,M,treatment,135.1,135.7,27.0,N,3,28871.0,Gwangju
P0062,53.0,M,control,150.1,147.0,16.5,N,5,15375.0,Daegu
P0063,37.0,M,control,144.8,136.7,23.1,N,1,17474.0,Gwangju
P0064,55.0,M,treatment,137.7,126.9,29.0,Y,4,15744.0,Busan
P0065,57.0,F,treatment,166.2,153.8,20.1,N,4,25969.0,Gwangju
P0066,55.0,F,treatment,145.1,137.3,24.0,Y,3,41480.0,Busan
P0067,51.0,F,control,140.6,136.7,25.2,N,4,9821.0,Busan
P0068,68.0,F,treatment,161.3,,24.2,N,2,21397.0,Gwangju
P0069,58.0,M,control,143.2,139.3,16.0,N,1,41523.0,Busan
P0070,86.0,F,control,145.2,146.3,27.0,Y,5,47069.0,Seoul
P0071,77.0,F,control,136.6,150.1,30.5,Y,5,11059.0,Seoul
P0072,69.0,M,control,157.1,156.2,24.5,Y,4,27664.0,Gwangju
P0073,47.0,M,control,156.2,155.2,20.7,Y,5,43633.0,Daegu
P0074,43.0,F,control,133.9,132.4,24.6,Y,4,28681.0,Daegu
P0075,62.0,F,treatment,132.1,125.4,32.7,N,2,24491.0,Busan
P0076,73.0,F,control,152.9,152.8,25.9,N,1,17686.0,Daegu
P0077,55.0,M,control,155.6,143.1,21.2,Y,4,7742.0,Daegu
P0078,51.0,M,treatment,140.6,127.6,26.8,N,3,34283.0,Seoul
P0079,52.0,F,treatment,153.8,141.9,25.0,Y,5,8775.0,Daegu
P0080,51.0,M,control,163.9,145.5,,Y,2,37663.0,Daegu
P0081,52.0,M,treatment,137.3,120.4,22.6,Y,2,22161.0,Seoul
P0082,41.0,F,control,159.0,145.6,20.8,N,4,54669.0,Daegu
P0083,53.0,M,control,135.9,137.9,30.1,N,1,30137.0,Seoul
P0084,49.0,M,control,152.6,150.7,32.6,Y,4,43018.0,Seoul
P0085,48.0,F,control,145.7,149.6,28.8,N,3,42605.0,Daegu
P0086,46.0,F,control,152.8,152.3,30.5,N,4,20892.0,Busan
P0087,69.0,M,treatment,168.0,146.2,28.4,Y,4,39125.0,Daegu
P0088,68.0,F,control,134.1,135.3,24.6,N,5,25873.0,Seoul
P0089,42.0,M,control,126.5,120.9,23.6,N,1,27999.0,Daegu
P0090,50.0,F,treatment,153.2,144.1,28.9,N,5,38156.0,Gwangju
P0091,53.0,M,control,167.5,157.8,24.4,Y,5,18802.0,Daegu
P0092, 47 ,F,control,143.4,139.1,30.8,N,2,49663.0,Seoul
P0093,55.0,M,control,127.7,123.9,20.6,Y,5,71117.0,Daegu
P0094,64.0,F,control,115.5,113.1,27.1,N,4,18509.0,Gwangju
P0095,69.0,M,control,156.0,159.8,26.6,N,3,44618.0,Daegu
P0096,50.0,F,treatment,165.2,153.3,25.9,Y,2,21331.0,Daegu
P0097,46.0,F,treatment,126.7,,25.2,N,4,26323.0,Busan
P0098,57.0,F,control,133.5,133.3,26.9,N,5,36500.0,Daegu
P0099,51.0,F,control,147.6,148.3,21.9,N,1,21461.0,Seoul
P0100,53.0,F,treatment,143.4,132.1,24.7,N,4,30403.0,Gwangju
P0101,71.0,F,treatment,157.0,153.9,24.9,N,2,32798.0,Busan
P0102,48.0,F,control,161.7,157.3,23.2,Y,3,47229.0,Busan
P0103,63.0,F,treatment,167.4,163.1,28.6,N,2,9597.0,Busan
P0104,56.0,F,treatment,136.4,117.3,27.9,N,4,27215.0,Daegu
P0105,62.0,F,control,144.3,141.1,26.9,N,5,40688.0,Daegu
P0106,41.0,F,treatment,162.2,151.1,30.6,N,4,25559.0,Seoul
P0107,54.0,F,treatment,144.2,128.9,26.7,N,4,37429.0,Daegu
P0108,28.0,F,control,148.0,143.9,23.8,N,5,40623.0,Daegu
P0109,59.0,F,control,147.8,147.4,23.4,N,1,25702.0,Daegu
P0110,54.0,F,control,168.9,175.6,26.7,Y,2,78180.0,Seoul
P0111,49.0,F,control,175.1,170.3,29.2,N,5,57014.0,Seoul
P0112,71.0,M,control,139.5,143.8,19.4,Y,4,13156.0,Gwangju
P0113,56.0,M,treatment,170.8,148.1,30.2,N,4,12769.0,Gwangju
P0114,66.0,F,control,130.1,130.0,24.4,N,1,41639.0,Gwangju
P0115,59.0,F,control,144.4,144.5,27.8,N,1,31583.0,Daegu
P0116,49.0,M,control,151.1,160.5,32.5,Y,2,95215.0,Gwangju
P0117,57.0,M,treatment,145.5,130.9,26.1,Y,1,25382.0,Daegu
P0118,52.0,F,control,137.3,,,N,4,23605.0,Daegu
P0119,31.0,F,control,132.7,141.8,24.8,N,2,6766.0,Seoul
P0120,64.0,F,treatment,119.6,101.6,29.3,N,5,22762.0,Daegu
P0121,72.0,M,control,165.0,156.9,23.2,N,1,12619.0,Gwangju
P0122,75.0,M,control,173.9,162.4,22.9,Y,4,11479.0,Seoul
P0123,49.0,M,control,142.6,135.5,24.1,Y,3,74950.0,Busan
P0124,66.0,F,treatment,167.8,154.9,19.7,N,3,49309.0,Daegu
P0125,37.0,F,control,168.1,163.2,27.1,N,2,37980.0,Busan
P0126,68.0,F,control,167.5,170.8,24.0,N,5,14773.0,Busan
P0127,55.0,F,treatment,119.6,,23.5,N,3,38653.0,Gwangju
P0128,70.0,F,control,153.7,147.5,25.6,N,5,15347.0,Daegu
P0129,63.0,F,treatment,175.8,154.4,22.3,N,1,21481.0,Gwangju
P0130,45.0,F,control,145.8,146.8,25.7,N,3,65265.0,Busan
P0131,51.0,M,treatment,157.3,150.8,25.0,N,5,11772.0,Daegu
P0132,49.0,F,treatment,146.1,136.6,30.7,N,4,35350.0,Daegu
P0133,61.0,M,control,161.7,157.6,23.8,Y,4,39063.0,Daegu
P0134,68.0,M,treatment,124.9,9.0,27.2,N,5,31211.0,Daegu
P0135,35.0,F,treatment,121.8,121.1,26.0,Y,2,26634.0,Daegu
P0136,46.0,M,treatment,146.1,128.2,29.0,N,2,18184.0,Seoul
P0137,59.0,F,treatment,164.9,170.6,22.3,N,2,14512.0,Seoul
P0138,47.0,M,treatment,146.2,145.5,22.6,N,3,38313.0,Gwangju
P0139,52.0,M,treatment,128.1,114.8,26.9,N,5,153003.0,Daegu
P0140,53.0,M,control,159.3,150.5,23.7,N,3,31681.0,Seoul
P0141,56.0,M,control,152.8,150.4,26.1,Y,1,11898.0,Busan
P0142,60.0,M,control,134.9,139.9,28.5,Y,3,14893.0,Gwangju
P0143,53.0,F,treatment,154.9,133.3,28.1,Y,2,26853.0,Seoul
P0144,59.0,M,treatment,173.3,161.8,22.0,N,1,26175.0,Daegu
P0145,40.0,M,treatment,128.9,,26.3,N,4,18401.0,Daegu
P0146,49.0,M,control,121.1,109.8,23.3,N,3,29908.0,Seoul
P0147,48.0,M,treatment,132.0,117.1,22.8,N,2,21847.0,Daegu
P0148,65.0,M,treatment,137.5,,27.4,Y,3,40174.0,Gwangju
P0149,51.0,F,treatment,154.0,152.7,21.5,N,2,20761.0,Daegu
P0150,61.0,F,control,156.5,147.6,19.7,Y,4,30679.0,Busan
P0151,29.0,M,control,134.7,129.5,29.5,Y,5,10778.0,Daegu
P0152,38.0,M,control,130.7,124.8,26.4,N,1,39255.0,Daegu
P0153,36.0,F,treatment,121.2,111.4,21.9,N,2,70491.0,Seoul
P0154,61.0,M,control,169.9,167.4,20.4,N,3,31538.0,Daegu
P0155,45.0,F,treatment,177.4,170.8,26.4,N,1,35181.0,Seoul
P0156,59.0,M,treatment,146.7,127.6,16.0,N,4,19414.0,Seoul
P0157,44.0,M,control,161.3,,,N,5,50047.0,Daegu
P0158,56.0,M,control,151.4,151.0,27.7,N,4,77464.0,Daegu
P0159,49.0,F,treatment,148.0,132.3,22.1,Y,3,28447.0,Seoul
P0160,38.0,F,control,126.6,125.9,26.6,N,1,25910.0,Seoul
P0161,63.0,F,treatment,137.1,116.3,32.5,N,1,59235.0,Seoul
P0162,57.0,F,treatment,141.5,126.4,29.3,N,5,25987.0,Gwangju
P0163,58.0,M,treatment,167.1,156.4,30.1,N,1,17583.0,Daegu
P0164,59.0,F,treatment,153.1,140.0,16.0,N,1,24298.0,Daegu
P0165,48.0,M,treatment,137.0,121.3,25.6,N,5,17519.0,Busan
P0166,55.0,F,control,168.1,161.8,24.3,N,1,43619.0,Busan
P0167,45.0,F,treatment,135.0,118.8,25.8,Y,2,22110.0,Daegu
P0168,60.0,M,treatment,147.4,135.6,29.3,N,4,30704.0,Daegu
P0169,37.0,F,control,135.1,128.0,32.3,N,4,16009.0,Seoul
P0170,76.0,M,control,172.8,175.9,26.5,N,3,23731.0,Daegu
P0171,60.0,M,treatment,153.8,136.3,23.7,N,4,18904.0,Daegu
P0172,58.0,F,control,130.9,125.2,22.3,Y,1,14560.0,Seoul
P0173,43.0,M,treatment,142.7,128.0,20.9,N,1,19676.0,Gwangju
P0174,74.0,F,treatment,138.1,130.7,24.6,N,1,22869.0,Seoul
P0175,61.0,F,control,145.6,129.1,22.7,Y,2,42134.0,Seoul
P0176,65.0,M,treatment,126.7,118.6,20.2,N,1,129301.0,Busan
P0177,62.0,M,treatment,150.3,136.0,27.8,N,3,55536.0,Daegu
P0178,62.0,F,treatment,146.7,134.0,20.3,N,2,24833.0,Daegu
P0179,70.0,F,control,153.7,141.8,30.0,N,1,20660.0,Seoul
P0180,54.0,F,treatment,143.6,130.4,29.4,Y,2,22577.0,Busan


In [ ]:
import pandas as pd
raw = pd.read_csv("exam_greenhouse.csv")     # 중간고사 데이터
print("온실 데이터", raw.shape)
raw.head(3)

---
## 1. 무엇을 규칙으로, 무엇을 LLM에게

| 규칙으로 고정 | LLM에게 위임 |
|---|---|
| 타입 추론 | 변수의 **의미** 파악 |
| 결측·이상치 탐지 | 사용자 질문 해석 |
| 검정의 전제 조건 | 결과를 문장으로 서술 |

**규칙으로 고정할 것의 두 가지 성질:**
1. 기계적으로 판정 가능하다
2. 틀리면 결과 자체가 무의미해진다

> 열이 몇 개의 고유값을 갖는지 세는 것은 기계가 정확합니다.
> LLM에게 물어보면 **느리고 비싸고 가끔 틀립니다.** 그냥 낭비입니다.

In [ ]:
from profiler import DataProfiler

prof = DataProfiler(raw)
print(prof.report())

> **LLM을 한 번도 부르지 않았습니다.** 그래서 빠르고, 공짜이고,
> 매번 같은 결과가 나옵니다.

In [ ]:
import time

t0 = time.perf_counter()
DataProfiler(raw)
규칙시간 = time.perf_counter() - t0

t0 = time.perf_counter()
_ = llm.ask(f"다음 데이터의 각 열 타입을 판정하라:\n{raw.head(5).to_string()}",
            temperature=0.0, max_tokens=300)
llm시간 = time.perf_counter() - t0

print(f"규칙 기반 : {규칙시간*1000:7.1f} ms")
print(f"LLM 호출  : {llm시간*1000:7.1f} ms   ({llm시간/규칙시간:.0f}배 느림)")
print()
print("그리고 LLM 은 매번 다른 답을 낼 수 있습니다.")

---
## 2. 두 가지 출력 — 사람용과 LLM용

In [ ]:
print("=== 사람용 report() ===")
print(prof.report()[:600])
print()
print("=== LLM용 to_prompt() ===")
print(prof.to_prompt())

In [ ]:
# 토큰 차이 (4주차 예산 문제)
from memory import estimate_tokens

print(f"report()    : {estimate_tokens(prof.report()):5d} 토큰")
print(f"to_prompt() : {estimate_tokens(prof.to_prompt()):5d} 토큰")
print()
print("열이 50개인 데이터의 전체 진단을 매 호출마다 보낼 수는 없습니다.")

---
## 3. 문제별 진단 결과 보기

In [ ]:
for name, 문제들 in prof.issues().items():
    print(f"[{name}]")
    for m in 문제들:
        print(f"   - {m}")
    print()

In [ ]:
# 개별 열의 상세 진단
p = prof.profiles["temp_c"]
print("이름:", p.name)
print("타입:", p.kind)
print("결측:", p.n_missing)
print("문제:", p.issues)
print("적용 가능한 정리:", p.fixes)
print("통계:", p.stats)

---
## 4. `clean()` — 무엇을 고쳤는지 전부 말한다

> **조용히 데이터를 바꾸는 것은 통계 분석에서 가장 위험한 일입니다.**
> 나중에 결과가 이상할 때 무엇 때문인지 추적할 수 없습니다.

In [ ]:
clean = prof.clean()

In [ ]:
# 정리 전후 비교
import numpy as np

비교 = []
for col in ["temp_c", "yield_after", "water_liters"]:
    전 = pd.to_numeric(raw[col], errors="coerce")
    후 = pd.to_numeric(clean[col], errors="coerce")
    비교.append({"열": col,
                "평균(전)": round(float(전.mean()), 2),
                "평균(후)": round(float(후.mean()), 2),
                "결측(전)": int(전.isna().sum()),
                "결측(후)": int(후.isna().sum())})
pd.DataFrame(비교)

In [ ]:
print("pest_damage")
print("  정리 전:", sorted(raw["pest_damage"].unique()))
print("  정리 후:", sorted(clean["pest_damage"].dropna().unique()))
print()
print("원본이 그대로인지 확인:", (raw["temp_c"] == -999).any())

---
## 5. 이상치는 기본적으로 건드리지 않는다

In [ ]:
# 기본: 알려만 준다
_ = DataProfiler(raw).clean(verbose=True)

In [ ]:
# 명시적으로 요청하면 제거
prof2 = DataProfiler(raw)
clean2 = prof2.clean(drop_outliers=True, verbose=True)

전 = pd.to_numeric(clean["yield_before"], errors="coerce")
후 = pd.to_numeric(clean2["yield_before"], errors="coerce")
print()
print(f"yield_before 최댓값  {전.max():.1f}  ->  {후.max():.1f}")
print(f"평균                {전.mean():.2f}  ->  {후.mean():.2f}")

> **이상치를 자동으로 제거하지 않는 이유**
>
> 혈압 480은 오류가 거의 확실하지만, 소득이 유난히 높은 사람은 진짜일 수 있습니다.
> **제거 여부는 분석자가 판단할 일이지 도구가 결정할 일이 아닙니다.**

---
## 6. 손대지 않는 경우 — 모르는 값이 섞였을 때

In [ ]:
# y/n 은 합칠 수 있을 것 같지만...
안전 = pd.DataFrame({"응답": ["Y", "y", "yes", "N", "no", "n"] * 10})
위험 = pd.DataFrame({"응답": ["Y", "y", "yes", "N", "no", "maybe"] * 10})

for 이름, d in [("전부 yes/no 계열", 안전), ("maybe 가 섞임", 위험)]:
    p = DataProfiler(d).profiles["응답"]
    print(f"{이름:18s} 수준 {sorted(d['응답'].unique())}")
    print(f"                   적용할 정리: {p.fixes}")
    print(f"                   진단: {p.issues}")
    print()

print("-> maybe 가 세 번째 범주인지, 결측인지, 오타인지 알 수 없습니다.")
print("   모르는 값이 섞여 있으면 손대지 않는 것이 안전합니다.")

---
## 연습문제

**문제 1.** `SYNONYM_FAMILIES` 에 성별 표기 집합이 있습니다.
`['M','F','남','여']` 가 섞인 열을 만들어 통합되는지 확인하시오.

**문제 2.** `ordinal_max_levels` 를 바꿔 가며 `satisfaction` 이
`ordinal` 로 판정되는 경계를 찾으시오.

**문제 3.** `DataProfiler` 에 새 진단 규칙을 추가하시오:
"두 열의 상관이 0.99 이상이면 중복 변수일 가능성" 을 알리는 것.

**문제 4.** 자신의 연구 데이터에 적용하고, 진단기가 **놓친 문제**가 있는지
직접 확인하시오. 있다면 어떤 규칙을 추가해야 하는가?

In [ ]:
# 여기에 직접 작성해 보세요

<details>
<summary><b>해설 보기</b></summary>

**문제 3.** 열 하나가 아니라 **열 쌍**을 보는 규칙이므로 지금 구조에 바로
안 들어갑니다. `ColumnProfile` 은 열 하나의 진단이기 때문입니다.

데이터셋 수준의 진단을 담을 자리가 따로 필요합니다.
```python
class DataProfiler:
    def __init__(self, df, ...):
        ...
        self.dataset_issues = []      # 열 쌍·전체에 관한 진단
        self._scan_pairs()

    def _scan_pairs(self):
        num = self.df.select_dtypes("number")
        corr = num.corr().abs()
        for i, a in enumerate(corr.columns):
            for b in corr.columns[i+1:]:
                if corr.loc[a, b] > 0.99:
                    self.dataset_issues.append(f"{a} 와 {b} 의 상관이 {corr.loc[a,b]:.3f}")
```
**설계를 확장할 때 자료구조가 맞는지 먼저 보는 것**이 요령입니다.
억지로 끼워 넣으면 나중에 더 아픕니다.

</details>

---
## 정리

- 타입 추론·결측 탐지는 **규칙이 LLM보다 낫다.** 빠르고 정확하고 공짜
- 출력을 **사람용 / LLM용**으로 나눈다 (토큰 예산)
- `clean()` 은 **무엇을 고쳤는지 전부 출력**한다
- **이상치는 알려만 준다.** 제거는 분석자의 판단
- **모르는 값이 섞이면 손대지 않는다**

**다음 노트북** → [`03_analysis_plan.ipynb`](https://colab.research.google.com/github/zoomer1975-boop/ai_agent/blob/main/week09/03_analysis_plan.ipynb)
이제 LLM에게 계획을 세우게 하고 규칙이 검증합니다.